# Classification Metrics
For the purposes of this notebook, we only use a single split instead of cross-validation.

In [ ]:
import pandas as pd

blood_transfusion = pd.read_csv("../transfusion.csv")
data = blood_transfusion.drop(columns="Class")
target = blood_transfusion["Class"]

In [ ]:
import matplotlib.pyplot as plt

target.value_counts().plot.barh()
plt.xlabel("Number of samples")
_ = plt.title("Number of samples per classes present\n in the target")

In [ ]:
from sklearn.model_selection import train_test_split

data_train, data_test, target_train, target_test = train_test_split(
    data, target, shuffle=True, random_state=0, test_size=0.5
)

In [ ]:
from sklearn.linear_model import LogisticRegression

classifier = LogisticRegression()
classifier.fit(data_train, target_train)

We create a synthetic sample for a new potential donor. Our classifier predicts they will not donate blood again.

In [ ]:
new_donor = pd.DataFrame(
    {
        "Recency": [6],
        "Frequency": [2],
        "Monetary": [1000],
        "Time": [20],
    }
)

In [ ]:
classifier.predict(new_donor)

## Accuracy as a Baseline
The accuracy is a baseline metric we use that calculates the mean percentage of correct predictions.

In [ ]:
from sklearn.metrics import accuracy_score

accuracy = accuracy_score(target_test, target_predicted)
print(f"Accuracy: {accuracy:.3f}")

## Confusion Matrix and Derived Metrics
The accuracy we calculated did not take into account the type of error the classifier was making. The **confusion matrix** gives us more granular details to identify false-negatives and false-positives.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

_ = ConfusionMatrixDisplay.from_estimator(classifier, data_test, target_test)

We have two key metrics that we can derive from this:
- **Precision Score**: `TP / (TP + FP)` or how likely the person actually gave blood when the classifier predicted they did
- **Recall Score**: `TP / (TP + FN)` or how well the classifier is able to predict people who did give blood

In [ ]:
from sklearn.metrics import precision_score, recall_score

precision = precision_score(target_test, target_predicted, pos_label=1)
recall = recall_score(target_test, target_predicted, pos_label=1)

print(f"Precision score: {precision:.3f}")
print(f"Recall score: {recall:.3f}")

The classifier did have a fair precision, however, it misclassified a lot of people who gave blood as having not donating blood.

In [ ]:
target_train.value_counts(normalize=True).plot.barh()
plt.xlabel("Class frequency")
_ = plt.title("Class frequency in the training set")

It can be seen that due to this class imbalance, it will naturally have a higher accuracy but a really low recall, as there are more samples in the negative class. A dummy classifier would achieve similar results if it scored on the most frequent class occurrence.

When classes are imbalanced, accuracy should not be used. Either precision or recall should be used, or the balanced accuracy should be used (which is the average recall obtained on each class).

In [ ]:
from sklearn.dummy import DummyClassifier

dummy_classifier = DummyClassifier(strategy="most_frequent")
dummy_classifier.fit(data_train, target_train)
print(
    "Accuracy of the dummy classifier: "
    f"{dummy_classifier.score(data_test, target_test):.3f}"
)

In [ ]:
from sklearn.metrics import balanced_accuracy_score

balanced_accuracy = balanced_accuracy_score(target_test, target_predicted)
print(f"Balanced accuracy: {balanced_accuracy:.3f}")

## Probability Thresholds
By default, the probability threshold to make a classification decision is 0.5. On the below precision-recall curve, each blue cross represents a level of probability which we used as a decision threshold. By varying this threshold, we get different precision vs. recall values.

A perfect classifier would have a precision of 1 for *all recall values*. The metric characterizing the curve is the **area under the curve (AUC)** and is named **average precision (AP)**. Thus, `AP = 1` for a perfect classifier.

In [ ]:
from sklearn.metrics import PrecisionRecallDisplay

disp = PrecisionRecallDisplay.from_estimator(
    classifier, data_test, target_test, pos_label=1, marker="+"
)
disp = PrecisionRecallDisplay.from_estimator(
    dummy_classifier,
    data_test,
    target_test,
    pos_label=1,
    color="tab:orange",
    linestyle="--",
    ax=disp.ax_,
)
plt.xlabel("Recall (also known as TPR or sensitivity)")
plt.ylabel("Precision (also known as PPV)")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
_ = disp.ax_.set_title("Precision-recall curve")

When we are interested on the negative class, we can use two other metrics:
- **Sensitivity**: the exact same as recall.
- **Specificity**: `TN / (TN + FP)` or the proportion of correctly classified samples in the negative class.

These can be plotted on the **Receiver Operating Characteristic (ROC) curve**:

In [ ]:
from sklearn.metrics import RocCurveDisplay

disp = RocCurveDisplay.from_estimator(
    classifier, data_test, target_test, pos_label=1, marker="+"
)
disp = RocCurveDisplay.from_estimator(
    dummy_classifier,
    data_test,
    target_test,
    pos_label=1,
    color="tab:orange",
    linestyle="--",
    ax=disp.ax_,
)
plt.xlabel("False positive rate")
plt.ylabel("True positive rate\n(also known as sensitivity or recall)")
plt.xlim(0, 1)
plt.ylim(0, 1)
plt.legend(bbox_to_anchor=(1.05, 0.8), loc="upper left")
_ = disp.ax_.set_title("Receiver Operating Characteristic curve")

In [ ]:
fig, axs = plt.subplots(ncols=2, nrows=1, figsize=(15, 7))

PrecisionRecallDisplay.from_estimator(
    classifier,
    data_test,
    target_test,
    pos_label=1,
    marker="+",
    plot_chance_level=True,
    chance_level_kw={"color": "tab:orange", "linestyle": "--"},
    ax=axs[0],
)
RocCurveDisplay.from_estimator(
    classifier,
    data_test,
    target_test,
    pos_label=1,
    marker="+",
    plot_chance_level=True,
    chance_level_kw={"color": "tab:orange", "linestyle": "--"},
    ax=axs[1],
)

_ = fig.suptitle("PR and ROC curves")